# 🎨 AI Image Generator: FLUX.1 Schnell
### 🛠️ Developed by: **Arafat Ahmed Mubin**
### 🌐 Brand: **2M**
---
**4-bit quantization + `device_map='auto'`** — runs FLUX.1-schnell on a free T4 GPU without OOM crashes.  
Fully compatible with **Google Colab** and **Kaggle**.

## 🛠️ Step 1: Environment Setup

In [ ]:
import os
# ── Environment Detection (Kaggle-First) ────────────────────────────────────────
if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
    ENV = "Kaggle"
else:
    try:
        import google.colab
        ENV = "Colab"
    except ImportError:
        ENV = "Local"

print(f"✅ Detected: {ENV}")
if ENV == "Kaggle":
    print("ℹ️  Kaggle: Ensure 'Internet' is ON in Notebook Settings.")

# ── Install Latest Packages ────────────────────────────────────────────────────
!pip install -q -U diffusers transformers accelerate bitsandbytes gradio
print("✅ All packages installed/updated.")

## 🧠 Step 2: Model Loading with 4-Bit Quantization
`BitsAndBytesConfig(load_in_4bit=True)` + `device_map='auto'` compresses FLUX.1 to ~8 GB and spreads it across VRAM, RAM, and CPU automatically.

In [ ]:
import torch, gc
from diffusers import FluxPipeline
from transformers import BitsAndBytesConfig
import gradio as gr

MODEL_ID = "black-forest-labs/FLUX.1-schnell"
current_pipe = None

def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def load_model():
    global current_pipe
    if current_pipe is not None:
        return current_pipe

    clear_memory()
    print(f"⏳ Loading {MODEL_ID} with 4-bit quantization…")

    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )

    current_pipe = FluxPipeline.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        device_map="auto",
        quantization_config=quant_config,
    )

    return current_pipe

# Pre-warm
load_model()

## 🎨 Step 3: Gradio Image Generation Interface

In [ ]:
import uuid
from PIL import Image

def generate_image(prompt, neg_prompt, width, height, steps, guidance, seed,
                   progress=gr.Progress(track_tqdm=True)):
    if not prompt.strip():
        return None, None, "❌ Prompt cannot be empty."
    try:
        pipe = load_model()
        gen  = torch.Generator("cpu")
        if int(seed) != -1:
            gen.manual_seed(int(seed))

        image = pipe(
            prompt=prompt,
            width=int(width), height=int(height),
            num_inference_steps=int(steps),
            guidance_scale=float(guidance),
            generator=gen,
            max_sequence_length=256,
        ).images[0]

        out_path = f"flux_{uuid.uuid4().hex[:8]}.png"
        image.save(out_path)
        clear_memory()
        return image, out_path, f"✅ Saved as {out_path}"
    except Exception as e:
        return None, None, f"❌ Error: {e}"

CSS = """
.gradio-container { font-family: 'Segoe UI', sans-serif; }
.gen-btn { background: linear-gradient(90deg, #f59e0b, #ef4444) !important; color: white !important; }
"""

with gr.Blocks(theme=gr.themes.Monochrome(), css=CSS, title="2M FLUX Image Studio") as demo:
    gr.Markdown("# 🎨 2M FLUX Image Studio\n**FLUX.1-schnell · 4-bit NF4 · device_map=auto · T4 Ready**")

    with gr.Row():
        with gr.Column(scale=1):
            prompt     = gr.Textbox(label="Prompt", lines=3, placeholder="A neon-lit cyberpunk samurai standing in the rain, ultra-detailed, 8k cinematic...")
            neg_prompt = gr.Textbox(label="Negative Prompt", placeholder="blurry, low quality, watermark, text")
            with gr.Row():
                width  = gr.Slider(256, 1360, value=1024, step=64, label="Width")
                height = gr.Slider(256, 1360, value=1024, step=64, label="Height")
            with gr.Row():
                steps    = gr.Slider(1, 30, value=4, step=1, label="Steps (Schnell: 4 is perfect)")
                guidance = gr.Slider(0.0, 10.0, value=0.0, step=0.1, label="Guidance Scale (0 for Schnell)")
            seed = gr.Number(label="Seed (-1 = random)", value=-1)
            btn  = gr.Button("✨ Generate Image", variant="primary", elem_classes="gen-btn")
            status = gr.Textbox(label="Status", interactive=False)
        with gr.Column(scale=1):
            out_img  = gr.Image(label="Generated Image", type="pil")
            out_file = gr.File(label="📥 Download PNG")

    btn.click(generate_image,
        inputs=[prompt, neg_prompt, width, height, steps, guidance, seed],
        outputs=[out_img, out_file, status])

demo.launch(share=True, debug=False)

## 📥 Step 4: View & Download Latest Image

In [ ]:
import glob, os
from IPython.display import Image as IPImage, display, FileLink

pngs = sorted(glob.glob("flux_*.png"), key=os.path.getmtime, reverse=True)
if pngs:
    latest = pngs[0]
    print(f"🖼  Showing: {latest}")
    display(IPImage(filename=latest, width=512))
    display(FileLink(latest, result_html_prefix="📥 Download: "))
else:
    print("No images yet — use the UI above to generate one!")

---
**© 2026 2M | All Rights Reserved**  
*Powered by the 2M Ecosystem (2M Dev's · 2M Future Facts · 2M Business Blogs)*